In [1]:
import pandas as pd


In [2]:
df = pd.read_csv("data/retail_sales_dataset.csv")

In [3]:
df.head()

,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    1000 non-null   int64 
 1   Date              1000 non-null   object
 2   Customer ID       1000 non-null   object
 3   Gender            1000 non-null   object
 4   Age               1000 non-null   int64 
 5   Product Category  1000 non-null   object
 6   Quantity          1000 non-null   int64 
 7   Price per Unit    1000 non-null   int64 
 8   Total Amount      1000 non-null   int64 
dtypes: int64(5), object(4)
memory usage: 70.4+ KB


In [5]:
df.describe()

,Transaction ID,Age,Quantity,Price per Unit,Total Amount
count,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000
mean,500.500000,41.39200,2.514000,179.890000,456.000000
std,288.819436,13.68143,1.132734,189.681356,559.997632
min,1.000000,18.00000,1.000000,25.000000,25.000000
25%,250.750000,29.00000,1.000000,30.000000,60.000000
50%,500.500000,42.00000,3.000000,50.000000,135.000000
75%,750.250000,53.00000,4.000000,300.000000,900.000000
max,1000.000000,64.00000,4.000000,500.000000,2000.000000


In [6]:
df = df.drop_duplicates()

In [7]:
df["Date"] = pd.to_datetime(df["Date"])

In [8]:
df.isnull().sum()

Transaction ID      0
Date                0
Customer ID         0
Gender              0
Age                 0
Product Category    0
Quantity            0
Price per Unit      0
Total Amount        0
dtype: int64

In [9]:
dim_customer = df[["Customer ID","Gender","Age"]].drop_duplicates()

In [10]:
dim_customer.head()

,Customer ID,Gender,Age
0,CUST001,Male,34
1,CUST002,Female,26
2,CUST003,Male,50
3,CUST004,Male,37
4,CUST005,Male,30


In [11]:
dim_product = df[["Product Category","Price per Unit"]].drop_duplicates()

In [12]:
dim_product["product_id"] = range(1, len(dim_product)+1)

In [13]:
dim_date = pd.DataFrame()
dim_date["full_date"] = df["Date"].drop_duplicates()
dim_date["month"] = dim_date["full_date"].dt.month
dim_date["year"] = dim_date["full_date"].dt.year

In [14]:
dim_date["date_id"] = range(1, len(dim_date)+1)

In [15]:
fact_sales = df.merge(dim_product,
                      on=["Product Category","Price per Unit"])

In [17]:
fact_sales.head()

,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount,product_id
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150,1
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000,2
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30,3
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500,2
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100,1


In [18]:
dim_date.head()

,full_date,month,year,date_id
0,2023-11-24,11,2023,1
1,2023-02-27,2,2023,2
2,2023-01-13,1,2023,3
3,2023-05-21,5,2023,4
4,2023-05-06,5,2023,5


In [19]:
fact_sales = fact_sales.merge(dim_date,
                              left_on="Date",
                              right_on="full_date")

In [20]:
fact_sales.head()

,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount,product_id,full_date,month,year,date_id
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150,1,2023-11-24,11,2023,1
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000,2,2023-02-27,2,2023,2
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30,3,2023-01-13,1,2023,3
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500,2,2023-05-21,5,2023,4
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100,1,2023-05-06,5,2023,5


In [21]:
fact_sales = fact_sales[
["Transaction ID",
 "date_id",
 "Customer ID",
 "product_id",
 "Quantity",
 "Total Amount"]
]

In [22]:
dim_customer.to_csv("dim_customer.csv", index=False)
dim_product.to_csv("dim_product.csv", index=False)
dim_date.to_csv("dim_date.csv", index=False)
fact_sales.to_csv("fact_sales.csv", index=False)